In [1]:
from IPython.core.getipython import get_ipython
from matplotlib import pyplot as plt
import numpy as np
import sys
import h5py
import os
import pandas as pd
sys.path.append("..")
from placecode import utils as ut
import pims_nd2
try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass
from datetime import datetime


sns.set(font_scale=3)
sns.set_style("whitegrid")

In [2]:
data_folder = ut.open_dir("Choose folder where all recordings are located!")

In [3]:
# first row should be header with column names: nd2, start, end
# then, the file name (without path), starting frame and end frame (inclusive-exclusive), where first frame is 1. 
# e.g. 240702_OPI2469_bl1_001.nd2	3182	3202  - python-indexed frames 3182 until 3201 (20 frames total) are read out of the
# file 240702_OPI2469_bl1_001.nd2 that should be somewhere (maybe in a subfolder) 
fpath_templates_excel = ut.open_file("Select excel file containing template file names and template begin-end frames!")

In [4]:
df_templates = pd.read_excel(fpath_templates_excel)

In [ ]:
fnames_nd2 = df_templates.nd2.unique()

In [ ]:
# check that all necessary nd2 files could be found
dict_fpaths_nd2 = dict()
for root, folders, files in os.walk(data_folder):
    for file in files:
        if file in fnames_nd2:
            assert os.path.exists(os.path.join(root, file))
            dict_fpaths_nd2[file] = os.path.join(root, file)

assert len(fnames_nd2) == len(dict_fpaths_nd2.keys())

In [ ]:
dict_segments = dict()
for i_row, row in df_templates.iterrows():
    i_begin = row["start"] - 1
    i_end = row["end"]
    with pims_nd2.ND2_Reader(dict_fpaths_nd2[row.nd2]) as nd2f:
        segment = np.array(nd2f[i_begin:i_end])
        dict_segments[row.nd2] = segment

d:\Software\anaconda\envs\placecoding\Lib\site-packages\pims\base_frames.py:478: UserWarning: Please call FramesSequenceND.__init__() at the start of thethe reader initialization.
  warn("Please call FramesSequenceND.__init__() at the start of the"


In [ ]:
dict_templates = dict()
for nd2 in dict_segments.keys():
    dict_templates[nd2] = np.mean(dict_segments[nd2], axis=0)  # dims: n_frames, x, y

In [ ]:
fpath_out = ut.open_dir("Select folder to save templates into!")

In [ ]:
fpath_export = os.path.join(fpath_out, "templates.hdf5")
with h5py.File(fpath_export, "w") as hf:
    for nd2 in dict_templates.keys():
            hf.create_dataset(name=nd2, data=dict_templates[nd2])
        
    